In [1]:
begin
    using Pkg
    dev_folder = @__DIR__
    Pkg.activate(dev_folder)
end

# ONLY run this block once to set up the development environment
# begin
#     pkg_folder = joinpath(dev_folder, "..")
#     Pkg.develop(path=pkg_folder)
#     Pkg.instantiate()
# end

Threads.nthreads()

  Activating project at `~/Documents/GitHub/Bnc_julia/Examples`


1

In [3]:
# using Revise
using BindingAndCatalysis # import the package
using CairoMakie # for plotting, we use Makie backend, could also be GLMakie or WGLMakie...

In [4]:
model = let
    N = [1 1 0 -1 0 0;
         0 1 1 0 -1 0;
         1 0 1 0 0 -1]  # define stoichiometry matrix
    x_sym = [:A, :B, :C, :ab, :bc, :ac] # Optional: define species symbols
    q_sym = [:tA, :tB, :tC] # Optional: define total concentration symbols
    K_sym = [:K12, :K23, :K13] # Optional: define binding constant symbols
    Bnc(N = N, x_sym=x_sym, q_sym=q_sym, K_sym=K_sym) # create Bnc model
end

----------Binding Network Summary:-------------
Number of species (n): 6
Number of conserved quantities (d): 3
Number of reactions (r): 3
L matrix: [1 0 … 0 1; 0 1 … 1 0; 0 0 … 1 1]
N matrix: [1 1 … 0 0; 0 1 … -1 0; 1 0 … 0 -1]
Direction of binding reactions: backward
Catalysis involved: No
Regimes constructed: No
-----------------------------------------------

In [5]:
find_all_vertices!(model) # find all possible vertices

┌ Info: ---------------------Start finding all vertices--------------------
└ @ BindingAndCatalysis /Users/wuxiaoyu/Documents/GitHub/Bnc_julia/src/regimes.jl:405
┌ Info: Finished, with 25 vertices found and 25 asymptotic vertices.
└ @ BindingAndCatalysis /Users/wuxiaoyu/Documents/GitHub/Bnc_julia/src/regimes.jl:414
┌ Info: -------------Start calculating nullity for each vertex, it also takes a while.------------
└ @ BindingAndCatalysis /Users/wuxiaoyu/Documents/GitHub/Bnc_julia/src/regimes.jl:415
┌ Info: 1.Building Nρ_inv cache in parallel...
└ @ BindingAndCatalysis /Users/wuxiaoyu/Documents/GitHub/Bnc_julia/src/regimes.jl:417
Progress: 100%|█████████████████████████████████████████| Time: 0:00:00
┌ Info: 2.Calculating nullity for each vertex in parallel...
└ @ BindingAndCatalysis /Users/wuxiaoyu/Documents/GitHub/Bnc_julia/src/regimes.jl:422


25-element Vector{Vector{Int8}}:
 [1, 2, 3]
 [1, 2, 5]
 [1, 2, 6]
 [1, 4, 3]
 [1, 4, 5]
 [1, 4, 6]
 [1, 5, 3]
 [1, 5, 5]
 [1, 5, 6]
 [4, 2, 3]
 ⋮
 [4, 5, 5]
 [6, 2, 3]
 [6, 2, 5]
 [6, 2, 6]
 [6, 4, 3]
 [6, 4, 6]
 [6, 5, 3]
 [6, 5, 5]
 [6, 5, 6]

In [6]:
summary(model) # Now regime data is available

----------Binding Network Summary:-------------
Number of species (n): 6
Number of conserved quantities (d): 3
Number of reactions (r): 3
L matrix: [1 0 0 1 0 1; 0 1 0 1 1 0; 0 0 1 0 1 1]
N matrix: [1 1 0 -1 0 0; 0 1 1 0 -1 0; 1 0 1 0 0 -1]
Direction of binding reactions: backward
Catalysis involved: No
Regimes constructed: Yes
Number of regimes: 25
  - Invertible + Asymptotic: 16
  - Singular +  Asymptotic: 9
  - Invertible +  Non-Asymptotic: 0
  - Singular +  Non-Asymptotic: 0
-----------------------------------------------


In [7]:
vtx_grh = get_vertices_graph!(model, full=true)

┌ Info: Start calculating vertices neighbor graph, It may takes a while.
└ @ BindingAndCatalysis /Users/wuxiaoyu/Documents/GitHub/Bnc_julia/src/regime_graphs.jl:463
Progress: 100%|█████████████████████████████████████████| Time: 0:00:00
┌ Info: Calculating vertices neighbor graph with qK change dir
└ @ BindingAndCatalysis /Users/wuxiaoyu/Documents/GitHub/Bnc_julia/src/regime_graphs.jl:420
Progress: 100%|█████████████████████████████████████████| Time: 0:00:00


BindingAndCatalysis.VertexGraph{Int8}(Bnc{Int8}([1 1 … 0 0; 0 1 … -1 0; 1 0 … 0 -1], [1 0 … 0 1; 0 1 … 1 0; 0 0 … 1 1], 3, 6, 3, Symbolics.Num[A, B, C, ab, bc, ac], Symbolics.Num[tA, tB, tC], Symbolics.Num[K12, K23, K13], nothing, Vector{Int8}[[1, 2, 3], [1, 2, 5], [1, 2, 6], [1, 4, 3], [1, 4, 5], [1, 4, 6], [1, 5, 3], [1, 5, 5], [1, 5, 6], [4, 2, 3]  …  [4, 5, 3], [4, 5, 5], [6, 2, 3], [6, 2, 5], [6, 2, 6], [6, 4, 3], [6, 4, 6], [6, 5, 3], [6, 5, 5], [6, 5, 6]], Dict{Vector{Int8}, Int64}([4, 4, 6] => 15, [4, 5, 5] => 17, [4, 2, 3] => 10, [1, 4, 6] => 6, [1, 5, 5] => 8, [1, 2, 3] => 1, [6, 5, 3] => 23, [4, 4, 3] => 13, [6, 2, 5] => 19, [1, 4, 3] => 4…), Bool[1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1, 1, 1, 1, 1, 1, 1, 1, 1, 1], Int8[0, 0, 0, 0, 0, 0, 0, 1, 0, 0  …  0, 1, 0, 0, 1, 0, 1, 0, 1, 1], BindingAndCatalysis.VertexGraph{Int8}(#= circular reference @-2 =#), BindingAndCatalysis.Vertex[BindingAndCatalysis.Vertex{Float64, Int8}(Bnc{Int8}(#= circular reference @-3 =#), Int8[1, 2, 3], 1, tru

In [8]:
siso = get_SISO_graph(model, :tA) # will give you a simple DiGraph

{25, 38} directed simple Int64 graph